### Creating new variables/columns into the dataset WITH DFS

In [1]:
import pandas as pd
import numpy as np

# pip install featuretools
import featuretools as ft

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import RFE, mutual_info_classif
from sklearn.model_selection import train_test_split

# load data
df = pd.read_csv("HeartDisease_Cleaned.csv")

df

c:\Users\Gayani\Advanced Data Analytics\AdvancedDataAnalytics\.venv\Lib\site-packages\woodwork\__init__.py:2: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


,Age,Sex,RestingBP,Cholesterol,FastingBS,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,HeartDisease,ChestPainType_ASY,ChestPainType_ATA,ChestPainType_NAP,ChestPainType_TA,RestingECG_LVH,RestingECG_Normal,RestingECG_ST
0,40,1,140,289,0,172,0,0.0,3,0,0,1,0,0,0,1,0
1,49,0,160,180,0,156,0,1.0,2,1,0,0,1,0,0,1,0
2,37,1,130,283,0,98,0,0.0,3,0,0,1,0,0,0,0,1
3,48,0,138,214,0,108,1,1.5,2,1,1,0,0,0,0,1,0
4,54,1,150,195,0,122,0,0.0,3,0,0,0,1,0,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
913,45,1,110,264,0,132,0,1.2,2,1,0,0,0,1,0,1,0
914,68,1,144,193,1,141,0,3.4,2,1,1,0,0,0,0,1,0
915,57,1,130,131,0,115,1,1.2,2,1,1,0,0,0,0,1,0
916,57,0,130,236,0,174,0,0.0,2,1,0,1,0,0,1,0,0


**DFS - Deep Feature Synthesis - automate generation of new columns via BRUTE FORCE**

By default, you should drop the target variable, because you can't have the target variable to be in any part of any new variable (because that is target leakage => not realistic from an actual ML model point of view).

In [4]:
# define the target variable
target = "HeartDisease"

# save for later use
y = df[target]

# create an entity set with the red wine data
es = ft.EntitySet(id="heart_disease")
es = es.add_dataframe(dataframe_name="heart_disease_data", 
                      dataframe=df.drop(target, axis=1),
                      index="index",
                      make_index=True)


# Use DFS to generate new variables (features), aggregate BASED on target variable: quality
feature_matrix, feature_defs = ft.dfs(
    entityset=es,
    target_dataframe_name="heart_disease_data",
    
    # agg_primitives=[],
    # if you use agg primitives, define also the calculations:
    # you can also use groupby trans primitives to nominal categories
    # groupby_trans_primitives=["cum_mean"],
    # trans_primitives=["multiply_numeric"],
    trans_primitives=["divide_numeric", "multiply_numeric"],
    # increase depth if you want to capture more complex
    # relationships in the data
    max_depth=1
)

# division a bit problematic, since often times
# values we divide with something very small (but not 0)
# this results in many rows and columns to go into infinity
# we have to take care of this

# THIS PART ONLY REQUIRED IF YOU DO DIVISION
# IT REMOVES ALL COLUMNS WITH INFINITE VALUES
feature_matrix.replace([np.inf, -np.inf], np.nan, inplace=True)
feature_matrix.dropna(axis=1, inplace=True)

# re-attach y back to feature_matrix
feature_matrix[target] = y


**After this, use RFE or something similar to see which variables to keep and which to drop!**

In [5]:
# typical X/y -split
X = feature_matrix.drop(target, axis=1)
y = feature_matrix[target]

# define model (random forest classifier in this example, because the target is 0 or 1)
# technically you can use pretty much any classic ML algorithm
# (random forest does not need scaled values, which keeps this example simple)
model = RandomForestClassifier(random_state=42)

# NOTE! LogisticRegression is faster, but it needs scaled values to converge
# model = LogisticRegression(max_iter=1000)

# create RFE, place the model and choose amount of optimal variables
rfe = RFE(estimator=model, n_features_to_select=12)

# fit the RFE model with our data
rfe.fit(X, y)

# get rankings and the results
rankings = rfe.ranking_
support = rfe.support_

results_df = pd.DataFrame({
    "Feature": X.columns,
    "Ranking": rankings,
    "Selected": support
}).sort_values(by="Ranking")

# if having lots of new variables, use Data Wrangler or
# something else to see all results
results_df.iloc[0:15]

,Feature,Ranking,Selected
8,ST_Slope,1,True
17,Age / ST_Slope,1,True
19,ChestPainType_ASY / MaxHR,1,True
56,ST_Slope / Age,1,True
59,Sex / MaxHR,1,True
60,Sex / ST_Slope,1,True
42,Oldpeak / MaxHR,1,True
46,RestingBP / ST_Slope,1,True
169,RestingBP * ST_Slope,1,True
158,MaxHR * ST_Slope,1,True


In [6]:
# compare the model performance before and after (as suggested above)
# NOTE: RFE has already seen all the rows, so this is only a rough comparison
from sklearn.model_selection import cross_val_score

selected = list(X.columns[rfe.support_])
original_features = numeric_cols + list(others.columns)

for name, cols in [("Original features", original_features), ("Selected by RFE", selected)]:
    scores = cross_val_score(RandomForestClassifier(random_state=42), X[cols], y, cv=5, scoring="accuracy")
    print(f"{name}: {len(cols)} features, accuracy {scores.mean():.4f}")

Original features: 16 features, accuracy 0.8289
Selected by RFE: 12 features, accuracy 0.8060
